In [3]:
def prune_weight_kernel_level(weight: torch.Tensor, sparsity: float) -> None:
    """커널 단위로 가중치를 프루닝하는 함수

    Args:
        weight: 프루닝할 가중치 텐서 (out_channels, in_channels, kernel_h, kernel_w)
        sparsity: 프루닝할 비율 (0~1 사이 값)

    Returns:
        프루닝 마스크 텐서
    """
    sparsity = min(1.0, max(0.0, sparsity))
    if sparsity == 1.0:
        weight.zero_()
        return torch.zeros_like(weight)
    elif sparsity == 0.0:
        return torch.ones_like(weight)

    # 프루닝할 커널 수 계산
    num_kernels = ### weight.shape[0] * weight.shape[1] ## Write Answer
    num_pruned_kernels = ### round(num_kernels * sparsity) ## Write Answer

    # 각 커널의 중요도를 절댓값 합으로 계산 (커널 크기에 대해 합산)
    importance = ### weight.abs().sum(dim=(2, 3), keepdim=True) ## Write Answer

    # pruning trheshold를 계산
    threshold = torch.kthvalue(importance.flatten(), num_pruned_kernels)[0]

    # threshold보다 큰 커널은 유지(1), 작은 커널은 제거(0)
    mask = importance > threshold

    # 마스크를 가중치와 동일한 크기로 확장
    mask = mask.expand_as(weight)

    # 마스크를 적용하여 프루닝 수행
    weight.mul_(mask)

    return mask

mask_kernel_level = prune_weight_kernel_level(weight.clone(), prune_sparsity)
draw_weight_distribution(mask_kernel_level, title="Kernel-level Pruning Mask")

SyntaxError: invalid syntax (3577984806.py, line 19)

In [4]:
### TODO : NGCF Layer 완성 ###

class NGCFLayer(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.1):
        super().__init__()
        self.W1 = nn.Linear(input_dim, output_dim)
        self.W2 = nn.Linear(input_dim, output_dim)
        # self.dropout = nn.Dropout(dropout)
        self.leaky_relu = nn.LeakyReLU(0.2) # jm - 0.2 -> 음수값 기울기

    def forward(self, edge_index, node_features, user_num, item_num):
        '''
        edge_index : 엣지 정보 (src, dst)의 집합.
        node_features: node별 이전 layer에서 생성된 벡터 정보가 담긴 matrix (H^(l-1)) (|V| * d)
        '''

        src, dst = edge_index # src : user, dst : movie

        # calculate node degree
        deg = torch.zeros(node_features.size(0), device=node_features.device)
        # calculate user degree
        deg.index_add_(0, src, torch.ones_like(src, dtype=torch.float))
        # jm - ex) [0,0,1,1,5,5,6]
        #          [1,1,1,1,1,1,1]
        #         deg[0] = 2
        #         deg[1] = 2
        #         deg[5] = 2
        #         deg[6] = 1
        # -> 이용자별 연결이 몇갠지 저장
       
        # calculate movie degree
        deg.index_add_(0, dst, torch.ones_like(dst, dtype=torch.float))

        # calculate 1/(root(deg(u)) * root(deg(i))) for all edge
        norm = ### 1.0/torch.sqrt(deg[src]*deg[dst]) ## Write Answer

        src_feat = node_features[src] # H_u
        dst_feat = node_features[dst] # H_i

        edge_messages_for_src = ### self.W1(dst_feat) + self.W2(dst_feat*src_feat) ## Write Answer
        edge_messages_for_src *= norm.unsqueeze(1)

        edge_messages_for_dst = ### self.W1(src_feat) + self.W2(src_feat*dst_feat) ## Write Answer
        edge_messages_for_dst *= norm.unsqueeze(1)

        aggregated_messages = torch.zeros_like(node_features)
        aggregated_messages.index_add(0, src, edge_messages_for_src)
        aggregated_messages[:user_num] += self.W1(node_features[:user_num])

        aggregated_messages.index_add_(0, dst, edge_messages_for_dst)
        aggregated_messages[user_num:] += self.W1(node_features[user_num:])

        aggregated_features = self.leaky_relu(aggregated_messages)

        # engineering approach
        # aggregated_features = self.dropout(aggregated_features)
        return aggregated_features

SyntaxError: invalid syntax (3365171293.py, line 35)